# Assignment 04 - Edge Computing & Intelligent Systems

**Q1 (50 Marks):** End-to-End Knowledge Distillation Framework

**Q2 (50 Marks):** Automated Neural Architecture Search for Edge Devices

---

**Runtime:** GPU (T4 or better recommended)

Run all cells in order from top to bottom.

## Install Dependencies

In [ ]:
!pip install -q torch torchvision onnx onnxruntime optuna timm
!pip install -q torchsummary psutil
print("All dependencies installed.")

## Shared Imports

In [ ]:
import os
import time
import copy
import json
import csv
import math
import random
import psutil
import traceback
import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, Subset

import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

import onnx
import onnxruntime as ort

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import warnings
warnings.filterwarnings('ignore')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)

os.makedirs('checkpoints', exist_ok=True)
os.makedirs('onnx_models', exist_ok=True)
os.makedirs('results', exist_ok=True)
print("Directories created: checkpoints/, onnx_models/, results/")

---
# QUESTION 1: Knowledge Distillation Framework
---

## Q1 - Section 1: CIFAR-100 Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# CIFAR-100 Data Loaders
# ---------------------------------------------------------------------------

CIFAR100_MEAN = (0.5071, 0.4867, 0.4408)
CIFAR100_STD  = (0.2675, 0.2565, 0.2761)

train_transform_c100 = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

test_transform_c100 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR100_MEAN, CIFAR100_STD),
])

cifar100_train = torchvision.datasets.CIFAR100(
    root='./data', train=True, download=True, transform=train_transform_c100
)
cifar100_test = torchvision.datasets.CIFAR100(
    root='./data', train=False, download=True, transform=test_transform_c100
)

train_loader_c100 = DataLoader(cifar100_train, batch_size=128, shuffle=True,
                               num_workers=2, pin_memory=True)
test_loader_c100  = DataLoader(cifar100_test,  batch_size=128, shuffle=False,
                               num_workers=2, pin_memory=True)

print(f"CIFAR-100 | Train: {len(cifar100_train)} | Test: {len(cifar100_test)}")

## Q1 - Section 2: Teacher Model (ResNet-50)

In [ ]:
# ---------------------------------------------------------------------------
# Teacher: ResNet-50 adapted for CIFAR-100 (32x32 images)
# ---------------------------------------------------------------------------

class TeacherResNet50(nn.Module):
    """ResNet-50 adapted for CIFAR-100 with a feature extraction hook."""

    def __init__(self, num_classes=100):
        super().__init__()
        base = models.resnet50(weights=None)
        # Replace the first conv to handle 32x32 input
        base.conv1 = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        base.maxpool = nn.Identity()
        base.fc = nn.Linear(base.fc.in_features, num_classes)
        self.backbone = base
        self._feature_map = None

        # Hook on layer3 output for feature-based distillation
        self.backbone.layer3.register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self._feature_map = output

    def forward(self, x):
        logits = self.backbone(x)
        return logits

    def get_features(self):
        return self._feature_map


def evaluate(model, loader, device):
    """Evaluate a model and return top-1 accuracy."""
    model.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, preds = outputs.max(1)
            correct += preds.eq(labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total


print("TeacherResNet50 class defined.")

In [ ]:
# ---------------------------------------------------------------------------
# Train Teacher Model
# ---------------------------------------------------------------------------

TEACHER_EPOCHS = 60
TEACHER_CKPT   = 'checkpoints/teacher_resnet50.pth'

teacher = TeacherResNet50(num_classes=100).to(DEVICE)

if os.path.exists(TEACHER_CKPT):
    teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=DEVICE))
    print(f"Loaded teacher checkpoint from {TEACHER_CKPT}")
    teacher_acc = evaluate(teacher, test_loader_c100, DEVICE)
    print(f"Teacher test accuracy: {teacher_acc:.2f}%")
else:
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(teacher.parameters(), lr=0.1, momentum=0.9,
                          weight_decay=5e-4, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=TEACHER_EPOCHS)

    best_acc = 0.0
    for epoch in range(1, TEACHER_EPOCHS + 1):
        teacher.train()
        running_loss = 0.0
        for images, labels in train_loader_c100:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(teacher(images), labels)
            loss.backward()
            optimizer.step()
            running_loss += loss.item()
        scheduler.step()

        if epoch % 10 == 0 or epoch == TEACHER_EPOCHS:
            acc = evaluate(teacher, test_loader_c100, DEVICE)
            avg_loss = running_loss / len(train_loader_c100)
            print(f"[Teacher] Epoch {epoch:3d}/{TEACHER_EPOCHS} | "
                  f"Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}%")
            if acc > best_acc:
                best_acc = acc
                torch.save(teacher.state_dict(), TEACHER_CKPT)

    # Load best weights
    teacher.load_state_dict(torch.load(TEACHER_CKPT, map_location=DEVICE))
    print(f"\nBest teacher test accuracy: {best_acc:.2f}%")

## Q1 - Section 3: Student Architectures

In [ ]:
# ---------------------------------------------------------------------------
# Student Model Definitions
#
# 1. MobileNetV2  - lightweight depthwise-separable convolutions
# 2. SqueezeNet   - fire modules for extreme compression
# 3. EfficientNet-Lite0 - compound-scaled efficient architecture
#
# All are adapted for 32x32 CIFAR-100 inputs and expose an
# intermediate feature map for feature-based distillation.
# ---------------------------------------------------------------------------

class StudentMobileNetV2(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        base = models.mobilenet_v2(weights=None)
        # Adapt first conv for 32x32
        base.features[0][0] = nn.Conv2d(3, 32, kernel_size=3,
                                         stride=1, padding=1, bias=False)
        base.classifier[-1] = nn.Linear(base.last_channel, num_classes)
        self.features = base.features
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.classifier = base.classifier
        self._feature_map = None
        # Hook at feature block 14 (mid-network)
        self.features[14].register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self._feature_map = output

    def forward(self, x):
        x = self.features(x)
        x = self.pool(x)
        x = x.flatten(1)
        return self.classifier(x)

    def get_features(self):
        return self._feature_map


class StudentSqueezeNet(nn.Module):
    def __init__(self, num_classes=100):
        super().__init__()
        base = models.squeezenet1_1(weights=None)
        # Replace first conv for 32x32
        base.features[0] = nn.Conv2d(3, 64, kernel_size=3,
                                     stride=1, padding=1)
        # Replace final conv
        base.classifier[1] = nn.Conv2d(512, num_classes, kernel_size=1)
        base.num_classes = num_classes
        self.features = base.features
        self.classifier = base.classifier
        self.pool = nn.AdaptiveAvgPool2d(1)
        self._feature_map = None
        self.features[9].register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self._feature_map = output

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        x = self.pool(x)
        return x.flatten(1)

    def get_features(self):
        return self._feature_map


class StudentEfficientNetLite0(nn.Module):
    """EfficientNet-Lite0 implemented from scratch as a lightweight
    compound-scaled network suitable for CIFAR-100 at 32x32."""

    class DepthwiseSeparable(nn.Module):
        def __init__(self, in_c, out_c, stride=1):
            super().__init__()
            self.dw = nn.Sequential(
                nn.Conv2d(in_c, in_c, 3, stride, 1, groups=in_c, bias=False),
                nn.BatchNorm2d(in_c), nn.ReLU6(inplace=True)
            )
            self.pw = nn.Sequential(
                nn.Conv2d(in_c, out_c, 1, bias=False),
                nn.BatchNorm2d(out_c), nn.ReLU6(inplace=True)
            )

        def forward(self, x):
            return self.pw(self.dw(x))

    class MBConv(nn.Module):
        def __init__(self, in_c, out_c, expand=6, stride=1):
            super().__init__()
            mid = in_c * expand
            self.use_residual = (stride == 1 and in_c == out_c)
            layers = []
            if expand != 1:
                layers += [nn.Conv2d(in_c, mid, 1, bias=False),
                           nn.BatchNorm2d(mid), nn.ReLU6(inplace=True)]
            layers += [
                nn.Conv2d(mid, mid, 3, stride, 1, groups=mid, bias=False),
                nn.BatchNorm2d(mid), nn.ReLU6(inplace=True),
                nn.Conv2d(mid, out_c, 1, bias=False),
                nn.BatchNorm2d(out_c)
            ]
            self.block = nn.Sequential(*layers)

        def forward(self, x):
            out = self.block(x)
            return x + out if self.use_residual else out

    def __init__(self, num_classes=100):
        super().__init__()
        MBConv = self.__class__.MBConv
        self.stem = nn.Sequential(
            nn.Conv2d(3, 32, 3, 1, 1, bias=False),
            nn.BatchNorm2d(32), nn.ReLU6(inplace=True)
        )
        self.blocks = nn.Sequential(
            MBConv(32,  16,  expand=1, stride=1),
            MBConv(16,  24,  expand=6, stride=1),
            MBConv(24,  24,  expand=6, stride=1),
            MBConv(24,  40,  expand=6, stride=2),
            MBConv(40,  40,  expand=6, stride=1),
            MBConv(40,  80,  expand=6, stride=2),
            MBConv(80,  80,  expand=6, stride=1),
            MBConv(80,  112, expand=6, stride=1),
            MBConv(112, 112, expand=6, stride=1),
            MBConv(112, 192, expand=6, stride=2),
            MBConv(192, 192, expand=6, stride=1),
            MBConv(192, 320, expand=6, stride=1),
        )
        self.head = nn.Sequential(
            nn.Conv2d(320, 1280, 1, bias=False),
            nn.BatchNorm2d(1280), nn.ReLU6(inplace=True)
        )
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(1280, num_classes)
        self._feature_map = None
        self.blocks[7].register_forward_hook(self._hook)

    def _hook(self, module, input, output):
        self._feature_map = output

    def forward(self, x):
        x = self.stem(x)
        x = self.blocks(x)
        x = self.head(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)

    def get_features(self):
        return self._feature_map


STUDENT_FACTORIES = {
    'MobileNetV2':       StudentMobileNetV2,
    'SqueezeNet':        StudentSqueezeNet,
    'EfficientNetLite0': StudentEfficientNetLite0,
}

print("Student model classes defined: MobileNetV2, SqueezeNet, EfficientNetLite0")

# Quick sanity check: forward pass for each student
dummy = torch.randn(2, 3, 32, 32)
for name, cls in STUDENT_FACTORIES.items():
    m = cls()
    out = m(dummy)
    print(f"  {name}: output shape {out.shape}  |  feature shape {m.get_features().shape}")
    del m

## Q1 - Section 4: Baseline Training (No Distillation)

In [ ]:
# ---------------------------------------------------------------------------
# Baseline Training Pipeline - No Knowledge Distillation
# ---------------------------------------------------------------------------

BASELINE_EPOCHS = 50

def train_baseline(model, name, train_loader, test_loader, epochs=BASELINE_EPOCHS):
    """Train a student model from scratch without knowledge distillation."""
    ckpt_path = f'checkpoints/baseline_{name}.pth'
    if os.path.exists(ckpt_path):
        model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        acc = evaluate(model, test_loader, DEVICE)
        print(f"[Baseline-{name}] Loaded checkpoint | Test Acc: {acc:.2f}%")
        return acc

    model = model.to(DEVICE)
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.SGD(model.parameters(), lr=0.05, momentum=0.9,
                          weight_decay=4e-5, nesterov=True)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        model.train()
        total_loss = 0.0
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if epoch % 10 == 0 or epoch == epochs:
            acc = evaluate(model, test_loader, DEVICE)
            avg_loss = total_loss / len(train_loader)
            print(f"[Baseline-{name}] Epoch {epoch:3d}/{epochs} | "
                  f"Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}%")
            if acc > best_acc:
                best_acc = acc
                torch.save(model.state_dict(), ckpt_path)

    model.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f"[Baseline-{name}] Best accuracy: {best_acc:.2f}%")
    return best_acc


baseline_results = {}
for sname, cls in STUDENT_FACTORIES.items():
    print(f"\n--- Baseline: {sname} ---")
    model = cls(num_classes=100).to(DEVICE)
    acc = train_baseline(model, sname, train_loader_c100, test_loader_c100)
    baseline_results[sname] = acc

print("\nBaseline Results:")
for k, v in baseline_results.items():
    print(f"  {k}: {v:.2f}%")

## Q1 - Section 5: Knowledge Distillation Framework

In [ ]:
# ---------------------------------------------------------------------------
# Knowledge Distillation Loss
#
# Supports:
#   - Soft-target KD loss (temperature-scaled KL divergence)
#   - Hard-label cross-entropy loss
#   - Feature-based distillation (L2 loss on intermediate feature maps)
#   - Configurable alpha (hard/soft balance) and temperature T
# ---------------------------------------------------------------------------

class FeatureAdapter(nn.Module):
    """1x1 conv to align student feature channels to teacher feature channels."""
    def __init__(self, student_channels, teacher_channels):
        super().__init__()
        self.adapt = nn.Conv2d(student_channels, teacher_channels,
                               kernel_size=1, bias=False)
        self.pool = nn.AdaptiveAvgPool2d(1)

    def forward(self, x):
        x = self.adapt(x)
        return self.pool(x).flatten(1)


class KnowledgeDistillationLoss(nn.Module):
    """
    Combined knowledge distillation loss:
        L = alpha * L_hard + (1 - alpha) * L_soft + beta * L_feature

    Args:
        temperature (float): Temperature for softening logits.
        alpha (float): Weight for hard-label cross-entropy.
        beta (float): Weight for feature distillation.
        teacher_feat_dim (int): Teacher intermediate feature channels.
        student_feat_dim (int): Student intermediate feature channels.
    """

    def __init__(self, temperature=4.0, alpha=0.3, beta=0.1,
                 teacher_feat_dim=1024, student_feat_dim=96):
        super().__init__()
        self.T = temperature
        self.alpha = alpha
        self.beta = beta
        self.ce = nn.CrossEntropyLoss()
        self.kl = nn.KLDivLoss(reduction='batchmean')
        self.mse = nn.MSELoss()
        self.adapter = FeatureAdapter(student_feat_dim, teacher_feat_dim)

    def soft_target_loss(self, student_logits, teacher_logits):
        """KL divergence between temperature-scaled softmax distributions."""
        s_soft = F.log_softmax(student_logits / self.T, dim=1)
        t_soft = F.softmax(teacher_logits / self.T, dim=1)
        return self.kl(s_soft, t_soft) * (self.T ** 2)

    def feature_loss(self, student_feat, teacher_feat):
        """Align and compute MSE between feature representations."""
        t_pooled = F.adaptive_avg_pool2d(teacher_feat, 1).flatten(1)
        s_adapted = self.adapter(student_feat)
        return self.mse(s_adapted, t_pooled.detach())

    def forward(self, student_logits, teacher_logits, labels,
                student_feat=None, teacher_feat=None):
        l_hard = self.ce(student_logits, labels)
        l_soft = self.soft_target_loss(student_logits, teacher_logits)
        loss = self.alpha * l_hard + (1 - self.alpha) * l_soft

        if student_feat is not None and teacher_feat is not None:
            l_feat = self.feature_loss(student_feat, teacher_feat)
            loss = loss + self.beta * l_feat

        return loss


print("KnowledgeDistillationLoss class defined.")

# Feature channel sizes for each student model
STUDENT_FEAT_DIMS = {
    'MobileNetV2':       96,
    'SqueezeNet':        256,
    'EfficientNetLite0': 112,
}
TEACHER_FEAT_DIM = 1024  # ResNet-50 layer3 output channels

print("Feature dimensions configured:")
for k, v in STUDENT_FEAT_DIMS.items():
    print(f"  {k}: student={v}, teacher={TEACHER_FEAT_DIM}")

## Q1 - Section 6: Train Students with Knowledge Distillation

In [ ]:
# ---------------------------------------------------------------------------
# Distillation Training Pipeline
# ---------------------------------------------------------------------------

KD_EPOCHS    = 50
TEMPERATURE  = 4.0
ALPHA        = 0.3   # hard-label weight
BETA         = 0.1   # feature distillation weight


def train_with_distillation(student, teacher, name, train_loader,
                             test_loader, epochs=KD_EPOCHS):
    """Train a student model using knowledge distillation from the teacher."""
    ckpt_path = f'checkpoints/kd_{name}.pth'
    if os.path.exists(ckpt_path):
        student.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
        acc = evaluate(student, test_loader, DEVICE)
        print(f"[KD-{name}] Loaded checkpoint | Test Acc: {acc:.2f}%")
        return acc

    student = student.to(DEVICE)
    teacher.eval()

    student_feat_dim = STUDENT_FEAT_DIMS[name]
    kd_criterion = KnowledgeDistillationLoss(
        temperature=TEMPERATURE, alpha=ALPHA, beta=BETA,
        teacher_feat_dim=TEACHER_FEAT_DIM, student_feat_dim=student_feat_dim
    ).to(DEVICE)

    params = list(student.parameters()) + list(kd_criterion.adapter.parameters())
    optimizer = optim.Adam(params, lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)

    best_acc = 0.0
    for epoch in range(1, epochs + 1):
        student.train()
        kd_criterion.train()
        total_loss = 0.0

        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)

            with torch.no_grad():
                teacher_logits = teacher(images)
                teacher_feat   = teacher.get_features()

            student_logits = student(images)
            student_feat   = student.get_features()

            loss = kd_criterion(
                student_logits, teacher_logits, labels,
                student_feat, teacher_feat
            )
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        scheduler.step()

        if epoch % 10 == 0 or epoch == epochs:
            acc = evaluate(student, test_loader, DEVICE)
            avg_loss = total_loss / len(train_loader)
            print(f"[KD-{name}] Epoch {epoch:3d}/{epochs} | "
                  f"Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}%")
            if acc > best_acc:
                best_acc = acc
                torch.save(student.state_dict(), ckpt_path)

    student.load_state_dict(torch.load(ckpt_path, map_location=DEVICE))
    print(f"[KD-{name}] Best accuracy: {best_acc:.2f}%")
    return best_acc


kd_results = {}
for sname, cls in STUDENT_FACTORIES.items():
    print(f"\n--- Distillation: {sname} ---")
    student = cls(num_classes=100).to(DEVICE)
    acc = train_with_distillation(
        student, teacher, sname, train_loader_c100, test_loader_c100
    )
    kd_results[sname] = acc

print("\nKnowledge Distillation Results:")
for k, v in kd_results.items():
    print(f"  {k}: {v:.2f}%")

## Q1 - Section 7: Comparison Table

In [ ]:
# ---------------------------------------------------------------------------
# Comparison: Baseline vs Knowledge Distillation
# ---------------------------------------------------------------------------

rows = []
for sname in STUDENT_FACTORIES:
    b_acc = baseline_results.get(sname, float('nan'))
    k_acc = kd_results.get(sname, float('nan'))
    gain  = k_acc - b_acc if not (math.isnan(b_acc) or math.isnan(k_acc)) else float('nan')
    rows.append({
        'Model': sname,
        'Baseline Acc (%)': round(b_acc, 2),
        'KD Acc (%)': round(k_acc, 2),
        'Gain (%)': round(gain, 2)
    })

df_q1 = pd.DataFrame(rows)
print("\n=== Q1 Results: Baseline vs Knowledge Distillation ===")
print(df_q1.to_string(index=False))

df_q1.to_csv('results/q1_comparison.csv', index=False)
print("\nSaved to results/q1_comparison.csv")

## Q1 - Section 8: Unified Inference Pipeline

In [ ]:
# ---------------------------------------------------------------------------
# Unified Inference Pipeline for all trained student models
# ---------------------------------------------------------------------------

def predict_student(model_name, image_tensor, variant='kd'):
    """
    Load a trained student checkpoint and run inference.

    Args:
        model_name (str): One of 'MobileNetV2', 'SqueezeNet', 'EfficientNetLite0'.
        image_tensor (Tensor): Preprocessed image of shape (1, 3, 32, 32).
        variant (str): 'kd' for distilled model, 'baseline' for baseline model.

    Returns:
        int: Predicted class index.
        Tensor: Class probabilities.
    """
    assert model_name in STUDENT_FACTORIES, f"Unknown model: {model_name}"
    assert variant in ('kd', 'baseline'), f"Unknown variant: {variant}"

    model = STUDENT_FACTORIES[model_name](num_classes=100).to(DEVICE)
    ckpt = f'checkpoints/{variant}_{model_name}.pth'
    if not os.path.exists(ckpt):
        raise FileNotFoundError(f"Checkpoint not found: {ckpt}")
    model.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    model.eval()

    with torch.no_grad():
        logits = model(image_tensor.to(DEVICE))
        probs  = F.softmax(logits, dim=1)
        pred   = probs.argmax(dim=1).item()

    return pred, probs.cpu()


# Demo: run inference on one test sample with every student model
sample_image, sample_label = cifar100_test[0]
sample_tensor = sample_image.unsqueeze(0)

print("Unified Inference Pipeline Demo")
print(f"  True label index: {sample_label}")
print()

for sname in STUDENT_FACTORIES:
    for variant in ['baseline', 'kd']:
        try:
            pred, probs = predict_student(sname, sample_tensor, variant=variant)
            confidence  = probs.max().item() * 100
            print(f"  [{variant:8s}] {sname:20s} | Pred: {pred:3d} | "
                  f"Conf: {confidence:.1f}%")
        except FileNotFoundError as e:
            print(f"  [{variant:8s}] {sname:20s} | {e}")

## Q1 - Section 9: ONNX Export and CPU Inference

In [ ]:
# ---------------------------------------------------------------------------
# ONNX Export for all distilled student models
# ---------------------------------------------------------------------------

def export_to_onnx(model, model_name, input_shape=(1, 3, 32, 32)):
    """Export a PyTorch model to ONNX format."""
    onnx_path = f'onnx_models/{model_name}.onnx'
    dummy_input = torch.randn(*input_shape, device='cpu')
    model_cpu = model.cpu().eval()

    torch.onnx.export(
        model_cpu,
        dummy_input,
        onnx_path,
        export_params=True,
        opset_version=11,
        do_constant_folding=True,
        input_names=['input'],
        output_names=['output'],
        dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
    )

    # Verify the exported model
    onnx_model = onnx.load(onnx_path)
    onnx.checker.check_model(onnx_model)

    size_mb = os.path.getsize(onnx_path) / (1024 ** 2)
    print(f"  Exported: {onnx_path} ({size_mb:.2f} MB)")
    return onnx_path


def onnx_cpu_inference(onnx_path, image_tensor):
    """Run CPU inference using ONNX Runtime."""
    sess_options = ort.SessionOptions()
    sess_options.intra_op_num_threads = 1
    session = ort.InferenceSession(
        onnx_path,
        sess_options=sess_options,
        providers=['CPUExecutionProvider']
    )
    input_name = session.get_inputs()[0].name
    input_data = image_tensor.numpy()

    start = time.perf_counter()
    output = session.run(None, {input_name: input_data})
    latency_ms = (time.perf_counter() - start) * 1000

    logits = output[0]
    pred   = int(np.argmax(logits, axis=1)[0])
    return pred, latency_ms


print("Exporting distilled student models to ONNX...")
onnx_benchmark_rows = []

for sname, cls in STUDENT_FACTORIES.items():
    ckpt = f'checkpoints/kd_{sname}.pth'
    if not os.path.exists(ckpt):
        print(f"  Skipping {sname}: checkpoint not found.")
        continue

    model = cls(num_classes=100)
    model.load_state_dict(torch.load(ckpt, map_location='cpu'))

    onnx_path = export_to_onnx(model, f'student_kd_{sname}')

    # CPU inference benchmark (average over 50 runs)
    sample = sample_tensor  # (1, 3, 32, 32)
    latencies = []
    for _ in range(50):
        pred, lat = onnx_cpu_inference(onnx_path, sample)
        latencies.append(lat)
    avg_lat = np.mean(latencies[5:])  # drop first 5 warmup runs

    size_mb = os.path.getsize(onnx_path) / (1024 ** 2)
    onnx_benchmark_rows.append({
        'Model': sname,
        'ONNX Path': onnx_path,
        'Size (MB)': round(size_mb, 2),
        'Avg CPU Latency (ms)': round(avg_lat, 3)
    })
    print(f"  {sname}: size={size_mb:.2f} MB | avg latency={avg_lat:.3f} ms")

df_onnx = pd.DataFrame(onnx_benchmark_rows)
df_onnx.to_csv('results/q1_onnx_benchmark.csv', index=False)
print("\nONNX benchmark results saved to results/q1_onnx_benchmark.csv")
print(df_onnx.to_string(index=False))

---
# QUESTION 2: Neural Architecture Search (NAS) for Edge Devices
---

## Q2 - Section 1: CIFAR-10 Data Loading

In [ ]:
# ---------------------------------------------------------------------------
# CIFAR-10 Data Loaders for NAS
# ---------------------------------------------------------------------------

CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

train_transform_c10 = transforms.Compose([
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

test_transform_c10 = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(CIFAR10_MEAN, CIFAR10_STD),
])

cifar10_train = torchvision.datasets.CIFAR10(
    root='./data', train=True, download=True, transform=train_transform_c10
)
cifar10_test = torchvision.datasets.CIFAR10(
    root='./data', train=False, download=True, transform=test_transform_c10
)

# Use a subset for fast NAS candidate evaluation
NAS_TRAIN_SIZE = 10000
NAS_VAL_SIZE   = 2000

nas_train_indices = list(range(NAS_TRAIN_SIZE))
nas_val_indices   = list(range(NAS_TRAIN_SIZE, NAS_TRAIN_SIZE + NAS_VAL_SIZE))

nas_train_loader = DataLoader(
    Subset(cifar10_train, nas_train_indices),
    batch_size=128, shuffle=True, num_workers=2, pin_memory=True
)
nas_val_loader = DataLoader(
    Subset(cifar10_train, nas_val_indices),
    batch_size=128, shuffle=False, num_workers=2, pin_memory=True
)

# Full loaders for final training after NAS
full_train_loader_c10 = DataLoader(
    cifar10_train, batch_size=128, shuffle=True, num_workers=2, pin_memory=True
)
full_test_loader_c10 = DataLoader(
    cifar10_test, batch_size=128, shuffle=False, num_workers=2, pin_memory=True
)

print(f"CIFAR-10 | Full train: {len(cifar10_train)} | Test: {len(cifar10_test)}")
print(f"NAS subset | Train: {NAS_TRAIN_SIZE} | Val: {NAS_VAL_SIZE}")

## Q2 - Section 2: Searchable Architecture Space

In [ ]:
# ---------------------------------------------------------------------------
# NAS Search Space Definition
#
# Architecture configuration dictionary covers:
#   - num_layers: variable convolutional depth (2-5 blocks)
#   - channels: variable number of filters per block
#   - kernel_sizes: 3 or 5
#   - use_depthwise: depthwise separable convolution blocks
#   - activation: relu, relu6, hardswish, silu
#   - input_resolution: 24 or 32 (simulated by cropping/padding)
# ---------------------------------------------------------------------------

NAS_SEARCH_SPACE = {
    'num_layers':        [2, 3, 4, 5],
    'channels':          [16, 32, 48, 64, 96],
    'kernel_sizes':      [3, 5],
    'use_depthwise':     [True, False],
    'activation':        ['relu', 'relu6', 'hardswish', 'silu'],
    'input_resolution':  [24, 32],
}

# Hardware constraints for edge deployment
MAX_LATENCY_MS    = 50.0   # Maximum allowed CPU inference latency
MAX_MODEL_SIZE_MB = 5.0    # Maximum allowed ONNX model size

print("NAS Search Space:")
for k, v in NAS_SEARCH_SPACE.items():
    print(f"  {k}: {v}")
print(f"\nHardware Constraints:")
print(f"  Max CPU latency : {MAX_LATENCY_MS} ms")
print(f"  Max model size  : {MAX_MODEL_SIZE_MB} MB")

## Q2 - Section 3: Architecture Generator

In [ ]:
# ---------------------------------------------------------------------------
# Architecture Generator
# Builds a CNN from a sampled architecture configuration dict.
# ---------------------------------------------------------------------------

ACTIVATION_MAP = {
    'relu':      nn.ReLU(inplace=True),
    'relu6':     nn.ReLU6(inplace=True),
    'hardswish': nn.Hardswish(inplace=True),
    'silu':      nn.SiLU(inplace=True),
}


class DepthwiseSepConvBlock(nn.Module):
    """Depthwise separable convolution block."""
    def __init__(self, in_c, out_c, kernel, activation):
        super().__init__()
        pad = kernel // 2
        self.block = nn.Sequential(
            # Depthwise
            nn.Conv2d(in_c, in_c, kernel, padding=pad, groups=in_c, bias=False),
            nn.BatchNorm2d(in_c),
            copy.deepcopy(activation),
            # Pointwise
            nn.Conv2d(in_c, out_c, 1, bias=False),
            nn.BatchNorm2d(out_c),
            copy.deepcopy(activation),
        )

    def forward(self, x):
        return self.block(x)


class StandardConvBlock(nn.Module):
    """Standard convolution block."""
    def __init__(self, in_c, out_c, kernel, activation):
        super().__init__()
        pad = kernel // 2
        self.block = nn.Sequential(
            nn.Conv2d(in_c, out_c, kernel, padding=pad, bias=False),
            nn.BatchNorm2d(out_c),
            copy.deepcopy(activation),
        )

    def forward(self, x):
        return self.block(x)


class NASNet(nn.Module):
    """
    Dynamically constructed CNN from an architecture configuration.

    Args:
        config (dict): Architecture configuration from the search space.
        num_classes (int): Output classes.
    """

    def __init__(self, config, num_classes=10):
        super().__init__()
        activation = ACTIVATION_MAP.get(config['activation'], nn.ReLU(inplace=True))
        channels   = config['channels']
        kernel     = config['kernel_sizes']
        use_dw     = config['use_depthwise']
        num_layers = config['num_layers']

        # Stem
        self.stem = nn.Sequential(
            nn.Conv2d(3, channels, 3, padding=1, bias=False),
            nn.BatchNorm2d(channels),
            copy.deepcopy(activation),
        )

        # Body: stacked conv blocks with max-pool for downsampling
        channel_schedule = [channels * min(2 ** i, 8) for i in range(num_layers)]
        # Cap at 256 to avoid memory explosion on small inputs
        channel_schedule = [min(c, 256) for c in channel_schedule]

        body = []
        in_c = channels
        for out_c in channel_schedule:
            if use_dw:
                body.append(DepthwiseSepConvBlock(in_c, out_c, kernel, activation))
            else:
                body.append(StandardConvBlock(in_c, out_c, kernel, activation))
            body.append(nn.MaxPool2d(2, 2))
            in_c = out_c
        self.body = nn.Sequential(*body)
        self.pool = nn.AdaptiveAvgPool2d(1)
        self.dropout = nn.Dropout(0.2)
        self.fc = nn.Linear(in_c, num_classes)
        self.config = config

    def forward(self, x):
        x = self.stem(x)
        x = self.body(x)
        x = self.pool(x).flatten(1)
        x = self.dropout(x)
        return self.fc(x)


def build_nas_model(config, num_classes=10):
    """Construct a NASNet from an architecture configuration."""
    return NASNet(config, num_classes=num_classes)


# Sanity check
test_config = {
    'num_layers': 3, 'channels': 32, 'kernel_sizes': 3,
    'use_depthwise': True, 'activation': 'relu6', 'input_resolution': 32
}
test_model = build_nas_model(test_config)
dummy = torch.randn(2, 3, 32, 32)
print(f"NASNet sanity check output: {test_model(dummy).shape}")
del test_model

## Q2 - Section 4: Hardware-Aware Measurement

In [ ]:
# ---------------------------------------------------------------------------
# Hardware-Aware Measurements
#   - CPU inference latency (median over multiple runs)
#   - ONNX model storage size
#   - Peak RSS memory during inference (via psutil)
# ---------------------------------------------------------------------------

def measure_cpu_latency(model, input_shape=(1, 3, 32, 32), n_runs=30):
    """Measure median CPU inference latency in milliseconds."""
    model = model.cpu().eval()
    dummy = torch.randn(*input_shape)
    latencies = []
    with torch.no_grad():
        for _ in range(n_runs):
            t0 = time.perf_counter()
            model(dummy)
            latencies.append((time.perf_counter() - t0) * 1000)
    return float(np.median(latencies))


def get_onnx_model_size(model, model_name, input_shape=(1, 3, 32, 32)):
    """Export model to ONNX and return storage size in MB."""
    onnx_path = f'onnx_models/nas_candidate_{model_name}.onnx'
    dummy_input = torch.randn(*input_shape)
    model = model.cpu().eval()
    try:
        torch.onnx.export(
            model, dummy_input, onnx_path,
            opset_version=11, export_params=True,
            do_constant_folding=True,
            input_names=['input'], output_names=['output'],
            dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
        )
        size_mb = os.path.getsize(onnx_path) / (1024 ** 2)
        return size_mb, onnx_path
    except Exception:
        return float('inf'), None


def measure_peak_inference_memory(model, input_shape=(1, 3, 32, 32)):
    """Measure peak RSS memory increase during a single forward pass (MB)."""
    process = psutil.Process(os.getpid())
    model = model.cpu().eval()
    dummy = torch.randn(*input_shape)
    mem_before = process.memory_info().rss
    with torch.no_grad():
        model(dummy)
    mem_after = process.memory_info().rss
    return max(0, (mem_after - mem_before) / (1024 ** 2))


def check_hardware_constraints(latency_ms, size_mb):
    """Return True if the architecture meets edge deployment constraints."""
    return (latency_ms <= MAX_LATENCY_MS) and (size_mb <= MAX_MODEL_SIZE_MB)


print("Hardware measurement utilities defined.")
print(f"  Deployment constraints: latency <= {MAX_LATENCY_MS} ms, "
      f"size <= {MAX_MODEL_SIZE_MB} MB")

## Q2 - Section 5: NAS Candidate Evaluation

In [ ]:
# ---------------------------------------------------------------------------
# Candidate Evaluation: quick training + validation + hardware measurement
# ---------------------------------------------------------------------------

NAS_QUICK_EPOCHS = 5   # Epochs per candidate during search (fast proxy)


def evaluate_candidate(config, trial_id, num_classes=10):
    """
    Train a candidate architecture for a few epochs, measure accuracy
    and hardware metrics, then return a composite NAS objective score.

    Returns:
        dict with keys: accuracy, latency_ms, size_mb, peak_mem_mb,
                        feasible, score
    """
    result = {
        'accuracy':    0.0,
        'latency_ms':  float('inf'),
        'size_mb':     float('inf'),
        'peak_mem_mb': 0.0,
        'feasible':    False,
        'score':       float('-inf'),
        'config':      config
    }

    try:
        model = build_nas_model(config, num_classes=num_classes).to(DEVICE)
    except Exception as e:
        print(f"  [Trial {trial_id}] Build error: {e}")
        return result

    # Quick training on NAS subset
    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-4)
    scheduler = optim.lr_scheduler.CosineAnnealingLR(
        optimizer, T_max=NAS_QUICK_EPOCHS
    )

    model.train()
    for _ in range(NAS_QUICK_EPOCHS):
        for images, labels in nas_train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        scheduler.step()

    # Validation accuracy
    acc = evaluate(model, nas_val_loader, DEVICE)
    result['accuracy'] = acc

    # CPU latency (input resolution from config)
    res = config.get('input_resolution', 32)
    latency = measure_cpu_latency(model, input_shape=(1, 3, res, res))
    result['latency_ms'] = latency

    # ONNX size
    size_mb, _ = get_onnx_model_size(
        model, f'trial_{trial_id}', input_shape=(1, 3, res, res)
    )
    result['size_mb'] = size_mb

    # Peak inference memory
    result['peak_mem_mb'] = measure_peak_inference_memory(
        model, input_shape=(1, 3, res, res)
    )

    # Constraint check
    feasible = check_hardware_constraints(latency, size_mb)
    result['feasible'] = feasible

    if feasible:
        # Score = accuracy penalized by normalized latency and size
        lat_penalty  = latency / MAX_LATENCY_MS
        size_penalty = size_mb / MAX_MODEL_SIZE_MB
        result['score'] = acc - 5.0 * lat_penalty - 3.0 * size_penalty
    else:
        # Infeasible: heavy penalty
        result['score'] = acc - 1000.0

    return result


print("Candidate evaluation function defined.")
print(f"  Quick train epochs per candidate: {NAS_QUICK_EPOCHS}")

## Q2 - Section 6: Bayesian Optimization NAS (Optuna)

In [ ]:
# ---------------------------------------------------------------------------
# NAS Search: Bayesian Optimization via Optuna
# ---------------------------------------------------------------------------

NAS_N_TRIALS  = 20   # Number of Optuna trials
search_history = []  # Store all trial results


def optuna_objective(trial):
    """Optuna objective function: sample a config, evaluate, return score."""
    config = {
        'num_layers':       trial.suggest_categorical('num_layers',       [2, 3, 4, 5]),
        'channels':         trial.suggest_categorical('channels',         [16, 32, 48, 64, 96]),
        'kernel_sizes':     trial.suggest_categorical('kernel_sizes',     [3, 5]),
        'use_depthwise':    trial.suggest_categorical('use_depthwise',    [True, False]),
        'activation':       trial.suggest_categorical('activation',       ['relu', 'relu6', 'hardswish', 'silu']),
        'input_resolution': trial.suggest_categorical('input_resolution', [24, 32]),
    }

    result = evaluate_candidate(config, trial_id=trial.number)
    result['trial'] = trial.number
    search_history.append(result)

    lat   = result['latency_ms']
    size  = result['size_mb']
    feas  = result['feasible']
    acc   = result['accuracy']
    score = result['score']
    print(f"  Trial {trial.number:3d} | Acc: {acc:.2f}% | "
          f"Lat: {lat:.1f}ms | Size: {size:.2f}MB | "
          f"Feasible: {feas} | Score: {score:.4f}")

    return score


print(f"Starting NAS with Optuna ({NAS_N_TRIALS} trials)...")
print(f"  Each trial trains for {NAS_QUICK_EPOCHS} epochs on "
      f"{NAS_TRAIN_SIZE} samples.")
print()

study = optuna.create_study(direction='maximize',
                             sampler=optuna.samplers.TPESampler(seed=SEED))
study.optimize(optuna_objective, n_trials=NAS_N_TRIALS, show_progress_bar=False)

best_trial  = study.best_trial
best_config = {k: v for k, v in best_trial.params.items()}

print(f"\nNAS complete. Best trial: {best_trial.number}")
print(f"Best config: {best_config}")
print(f"Best score : {best_trial.value:.4f}")

## Q2 - Section 7: Save Search History

In [ ]:
# ---------------------------------------------------------------------------
# Save NAS search history to CSV and JSON
# ---------------------------------------------------------------------------

history_rows = []
for r in search_history:
    row = {
        'trial':       r.get('trial', -1),
        'accuracy':    round(r['accuracy'], 4),
        'latency_ms':  round(r['latency_ms'], 4),
        'size_mb':     round(r['size_mb'], 4),
        'peak_mem_mb': round(r['peak_mem_mb'], 4),
        'feasible':    r['feasible'],
        'score':       round(r['score'], 4),
    }
    # Flatten config into the row
    for k, v in r['config'].items():
        row[f'cfg_{k}'] = v
    history_rows.append(row)

df_history = pd.DataFrame(history_rows).sort_values('score', ascending=False)
df_history.to_csv('results/q2_nas_search_history.csv', index=False)

with open('results/q2_best_config.json', 'w') as f:
    json.dump({'best_config': best_config, 'best_score': best_trial.value}, f, indent=2)

print("Saved NAS results:")
print("  results/q2_nas_search_history.csv")
print("  results/q2_best_config.json")
print()
print("Top 5 candidates by score:")
print(df_history.head(5)[['trial','accuracy','latency_ms','size_mb','feasible','score']].to_string(index=False))

## Q2 - Section 8: Train Best Discovered Architecture from Scratch

In [ ]:
# ---------------------------------------------------------------------------
# Train the best NAS architecture from scratch on full CIFAR-10
# ---------------------------------------------------------------------------

NAS_FINAL_EPOCHS = 60
NAS_BEST_CKPT    = 'checkpoints/nas_best_model.pth'

best_nas_model = build_nas_model(best_config, num_classes=10).to(DEVICE)
print("Best architecture config:")
for k, v in best_config.items():
    print(f"  {k}: {v}")
print()

if os.path.exists(NAS_BEST_CKPT):
    best_nas_model.load_state_dict(torch.load(NAS_BEST_CKPT, map_location=DEVICE))
    nas_final_acc = evaluate(best_nas_model, full_test_loader_c10, DEVICE)
    print(f"Loaded NAS best model checkpoint | Test Acc: {nas_final_acc:.2f}%")
else:
    criterion  = nn.CrossEntropyLoss()
    optimizer  = optim.SGD(best_nas_model.parameters(), lr=0.1, momentum=0.9,
                            weight_decay=5e-4, nesterov=True)
    scheduler  = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=NAS_FINAL_EPOCHS)

    best_acc_nas = 0.0
    for epoch in range(1, NAS_FINAL_EPOCHS + 1):
        best_nas_model.train()
        total_loss = 0.0
        for images, labels in full_train_loader_c10:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            loss = criterion(best_nas_model(images), labels)
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        scheduler.step()

        if epoch % 10 == 0 or epoch == NAS_FINAL_EPOCHS:
            acc = evaluate(best_nas_model, full_test_loader_c10, DEVICE)
            avg_loss = total_loss / len(full_train_loader_c10)
            print(f"[NAS-Final] Epoch {epoch:3d}/{NAS_FINAL_EPOCHS} | "
                  f"Loss: {avg_loss:.4f} | Test Acc: {acc:.2f}%")
            if acc > best_acc_nas:
                best_acc_nas = acc
                torch.save(best_nas_model.state_dict(), NAS_BEST_CKPT)

    best_nas_model.load_state_dict(torch.load(NAS_BEST_CKPT, map_location=DEVICE))
    nas_final_acc = best_acc_nas
    print(f"\nNAS best model final test accuracy: {nas_final_acc:.2f}%")

## Q2 - Section 9: ONNX Export and Inference Pipeline for NAS Model

In [ ]:
# ---------------------------------------------------------------------------
# Export the best NAS model to ONNX and benchmark CPU inference
# ---------------------------------------------------------------------------

NAS_RES     = best_config.get('input_resolution', 32)
NAS_IN_SHAPE = (1, 3, NAS_RES, NAS_RES)

nas_onnx_path = f'onnx_models/nas_best_model.onnx'
dummy_input   = torch.randn(*NAS_IN_SHAPE, device='cpu')
best_nas_model_cpu = best_nas_model.cpu().eval()

torch.onnx.export(
    best_nas_model_cpu,
    dummy_input,
    nas_onnx_path,
    export_params=True,
    opset_version=11,
    do_constant_folding=True,
    input_names=['input'],
    output_names=['output'],
    dynamic_axes={'input': {0: 'batch_size'}, 'output': {0: 'batch_size'}}
)

# Verify
onnx_model = onnx.load(nas_onnx_path)
onnx.checker.check_model(onnx_model)
nas_size_mb = os.path.getsize(nas_onnx_path) / (1024 ** 2)
print(f"Exported NAS model: {nas_onnx_path} ({nas_size_mb:.2f} MB)")


def nas_onnx_inference(onnx_path, input_shape):
    """Run ONNX Runtime CPU inference and return latency."""
    sess = ort.InferenceSession(
        onnx_path, providers=['CPUExecutionProvider']
    )
    input_name = sess.get_inputs()[0].name
    dummy = np.random.randn(*input_shape).astype(np.float32)
    # Warmup
    for _ in range(5):
        sess.run(None, {input_name: dummy})
    # Benchmark
    latencies = []
    for _ in range(50):
        t0 = time.perf_counter()
        out = sess.run(None, {input_name: dummy})
        latencies.append((time.perf_counter() - t0) * 1000)
    return float(np.median(latencies)), out[0]


nas_lat_ms, nas_output = nas_onnx_inference(nas_onnx_path, NAS_IN_SHAPE)
nas_pred = int(np.argmax(nas_output, axis=1)[0])

print(f"NAS ONNX CPU inference latency: {nas_lat_ms:.3f} ms")
print(f"Sample prediction (class index): {nas_pred}")
print(f"Hardware constraints satisfied: "
      f"{check_hardware_constraints(nas_lat_ms, nas_size_mb)}")

# Save NAS ONNX benchmark
nas_onnx_result = {
    'Model': 'NAS Best Model',
    'Config': str(best_config),
    'Test Accuracy (%)': round(nas_final_acc, 2),
    'ONNX Size (MB)': round(nas_size_mb, 2),
    'CPU Latency (ms)': round(nas_lat_ms, 3),
    'Feasible': check_hardware_constraints(nas_lat_ms, nas_size_mb)
}
with open('results/q2_onnx_benchmark.json', 'w') as f:
    json.dump(nas_onnx_result, f, indent=2)

print("\nNAS ONNX benchmark saved to results/q2_onnx_benchmark.json")

## Q2 - Section 10: Benchmarking Summary

In [ ]:
# ---------------------------------------------------------------------------
# Automated Benchmarking: compare all models
# ---------------------------------------------------------------------------

print("=" * 70)
print("FINAL BENCHMARKING SUMMARY")
print("=" * 70)

bench_rows = []

# Q1 Student models (distilled)
for sname, cls in STUDENT_FACTORIES.items():
    ckpt = f'checkpoints/kd_{sname}.pth'
    if not os.path.exists(ckpt):
        continue
    model = cls(num_classes=100)
    model.load_state_dict(torch.load(ckpt, map_location='cpu'))
    lat = measure_cpu_latency(model)
    size, _ = get_onnx_model_size(model, f'bench_{sname}')
    acc = kd_results.get(sname, float('nan'))
    bench_rows.append({
        'Task': 'Q1-KD',
        'Model': sname,
        'Dataset': 'CIFAR-100',
        'Test Acc (%)': round(acc, 2),
        'CPU Latency (ms)': round(lat, 3),
        'ONNX Size (MB)': round(size, 2),
    })

# Q2 NAS best model
bench_rows.append({
    'Task': 'Q2-NAS',
    'Model': 'NAS Best',
    'Dataset': 'CIFAR-10',
    'Test Acc (%)': round(nas_final_acc, 2),
    'CPU Latency (ms)': round(nas_lat_ms, 3),
    'ONNX Size (MB)': round(nas_size_mb, 2),
})

df_bench = pd.DataFrame(bench_rows)
print(df_bench.to_string(index=False))
df_bench.to_csv('results/final_benchmark.csv', index=False)
print("\nFinal benchmark saved to results/final_benchmark.csv")

## Final: List All Output Files

In [ ]:
# ---------------------------------------------------------------------------
# List all generated checkpoints, ONNX models, and result files
# ---------------------------------------------------------------------------

print("=" * 60)
print("ALL GENERATED OUTPUT FILES")
print("=" * 60)

for folder in ['checkpoints', 'onnx_models', 'results']:
    print(f"\n{folder}/")
    if os.path.exists(folder):
        for fname in sorted(os.listdir(folder)):
            fpath = os.path.join(folder, fname)
            size  = os.path.getsize(fpath) / (1024 ** 2)
            print(f"  {fname:<50s}  {size:.2f} MB")
    else:
        print("  (empty)")

print("\nAll tasks complete.")